## Capstone Walk-Forward Validation — Rolling Train->Test Folds

Extends history to ~25 years (2001-2026, OEF ETF inception) and replaces the
single 50/50 split with 5 rolling, expanding-window folds. Each fold fits causal
discovery on TRAIN only and tests on the next untouched segment, so the strategy
is checked across several different market regimes instead of one lucky split.

v4's thresholds (baseline 2.0, floor 1.3, calm cutoff 1.8, 10-day EMA, 252-day
rolling percentile, 5-day re-entry) are frozen — not re-tuned per fold.

**Same caveat as before:** the AR-GARCH step (Section 3) is still fit once on
the full sample, not re-fit per fold. Scope limitation, not fixed here.


### Section 1-3: Data ingestion (extended to ~25y), stationarity, AR-GARCH

In [ ]:
# ==============================================================================
# SECTION 1.1: DEPENDENCY MANAGEMENT & GLOBAL REPRODUCIBILITY
# ==============================================================================
import os
import sys
import random
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
import requests

# Suppress annoying, non-critical package warnings for clean academic presentation
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

def seed_everything(seed: int = 42):
    """
    Enforces strict deterministic behavior across random processes,
    shuffles, and statistical bootstraps to ensure cross-machine verification.
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)
print("✅ Section 1.1 Success: Environment initialized and random states locked.")

# ==============================================================================
# SECTION 1.2: HIGH-FIDELITY DATA INGESTION PIPELINE
# ==============================================================================

# 🔴 CONFIGURATION: Read your FRED API key from an environment variable instead
# of hardcoding it in the notebook. Set it before running, e.g.:
#   export FRED_API_KEY="your_32_char_key"
# If unset, the engine automatically switches to Web-Scrape Fallback Mode.
FRED_API_KEY = os.environ.get("FRED_API_KEY", "")

def fetch_systematic_risk_data(start_date: str, end_date: str) -> pd.DataFrame:
    """
    Ingests high-fidelity equity index parameters and macroeconomic risk features.

    Academic Defenses:
    1. Proxy Integrity: Uses 'OEF' (iShares S&P 100 ETF) to account for transactional
       frictions and tracking errors absent in theoretical index benchmarks.
    2. Feature Mapping: Captures implied risk via the VIX index, structural credit
       stress via the BofA High-Yield Spread, and market liquidity via the Volume-to-Float ratio.
    """
    print(f"📥 Initiating data ingestion window: {start_date} to {end_date}...")

    # --------------------------------------------------------------------------
    # Part A: Financial Market Data Extraction (Yahoo Finance)
    # --------------------------------------------------------------------------
    tickers = {'SP100_Price': 'OEF', 'VIX': '^VIX'}
    print("   👉 Pulling equity metrics from Yahoo Finance API...")

    market_data = yf.download(
        list(tickers.values()),
        start=start_date,
        end=end_date,
        interval='1d',
        auto_adjust=True,
        progress=False
    )

    # Securely flatten and map multi-index columns
    df_market = pd.DataFrame(index=market_data.index)
    df_market['SP100_Close']  = market_data['Close']['OEF']
    df_market['SP100_Open']   = market_data['Open']['OEF']
    df_market['SP100_High']   = market_data['High']['OEF']
    df_market['SP100_Low']    = market_data['Low']['OEF']
    df_market['SP100_Volume'] = market_data['Volume']['OEF']
    df_market['VIX_Close']    = market_data['Close']['^VIX']

    # --------------------------------------------------------------------------
    # Part B: Macroeconomic Risk Ingestion (FRED API with Multi-Tier Fallbacks)
    # --------------------------------------------------------------------------
    series_id = "BAMLEM251413057LOC"  # ICE BofA High Yield Option-Adjusted Spread
    df_fred = pd.DataFrame()

    # Mode 1: Authentication via Official JSON API
    # 🔧 FIX: the previous URL pointed at the bare FRED homepage with no API
    # path, so it never actually reached the observations endpoint. This is
    # the real endpoint for pulling a single series as JSON.
    if FRED_API_KEY:
        print("   👉 Authenticating with St. Louis FRED via JSON API endpoint...")
        fred_url = "https://api.stlouisfed.org/fred/series/observations"
        params = {
            'series_id': series_id,
            'api_key': FRED_API_KEY,
            'file_type': 'json',
            'observation_start': start_date,
            'observation_end': end_date
        }
        try:
            response = requests.get(fred_url, params=params, timeout=15)
            if response.status_code == 200:
                json_data = response.json()
                obs = json_data['observations']
                df_fred = pd.DataFrame(obs)
                df_fred['date'] = pd.to_datetime(df_fred['date'])
                df_fred.set_index('date', inplace=True)
                df_fred['Credit_Spread'] = pd.to_numeric(df_fred['value'], errors='coerce')
                df_fred = df_fred[['Credit_Spread']]
                print("      ✅ Authenticated JSON query successful.")
            else:
                print(f"      ⚠️ API Error (Status {response.status_code}). Switching to Fallback Mode...")
        except Exception as e:
            print(f"      ⚠️ Request failed ({str(e)}). Switching to Fallback Mode...")
    else:
        print("   👉 No FRED_API_KEY set. Skipping Mode 1 and trying the unauthenticated fallback...")

    # Mode 2: Unauthenticated Web-Scrape Fallback (Direct CSV stream parsing)
    # 🔧 FIX: the previous URL concatenated the series id straight onto the bare
    # domain (an invalid hostname). This is FRED's actual public CSV graph endpoint.
    if df_fred.empty:
        print("   👉 Mode 2: Direct web-stream parsing activated (No API key required)...")
        csv_fallback_url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
        try:
            df_csv = pd.read_csv(csv_fallback_url, parse_dates=['DATE'], index_col='DATE')
            df_csv[series_id] = pd.to_numeric(df_csv[series_id], errors='coerce')
            df_fred = df_csv.rename(columns={series_id: 'Credit_Spread'})
            df_fred = df_fred.loc[start_date:end_date]
            print("      ✅ Web-stream parsing successful.")
        except Exception as e:
            print(f"      ❌ Mode 2 Failure: Server blocked raw query ({str(e)}). Activating Mode 3...")

    # Mode 3: Local Parametric Simulation (Ensures code compile during live defense)
    if df_fred.empty:
        print("   ⚠️ Mode 3: Generating a synthetic econometric proxy matrix...")
        df_fred = pd.DataFrame(index=df_market.index)
        # Emulates high-yield spread profiles (Mean=4.0%, SD=0.5%) matching standard distributions
        df_fred['Credit_Spread'] = np.random.normal(4.0, 0.5, len(df_market))
        print("      ✅ Local econometric matrix synthesized.")
        print("      ⚠️ NOTE: This is synthetic noise, not real credit-spread data — any causal")
        print("         pathway involving Credit_Spread_Diff downstream is not economically meaningful")
        print("         until Mode 1 or Mode 2 succeeds.")

    # Join structural dataframes using a left outer execution map on trading days
    raw_df = df_market.join(df_fred, how='left')
    return raw_df

# Execute ingestion across a robust ~25-year historical window (OEF ETF inception is Oct 2000)
raw_data = fetch_systematic_risk_data(start_date="2001-01-01", end_date="2026-01-01")
print(f"📊 Section 1.2 Success: Raw Matrix compiled. Dimensions: {raw_data.shape}")

# ==============================================================================
# SECTION 1.3: DATA ALIGNMENT, BOUNDARY HANDLING & VOLATILITY COMPUTATION
# ==============================================================================
def process_and_align_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans structural alignment artifacts and engineers advanced variance parameters.

    Academic Defenses against Look-Ahead and Frequency Contamination:
    1. Multi-Frequency Isolation: Drops calendar days without transactional stock
       history and forward-fills macro variables to match data release lags safely.
    2. Variance Efficiency: Replaces simple close-to-close measures with Garman-Klass
       volatility estimators. Incorporating opening gaps and intraday extremes provides
       an estimation efficiency gains up to 8x higher.
    """
    print("\n⚙️ Running alignment corrections and variance calculations...")
    cleaned_df = df.copy()

    # Eliminate non-trading calendar frames (weekends and national market closures)
    cleaned_df = cleaned_df[cleaned_df['SP100_Close'].notna() & (cleaned_df['SP100_Volume'] > 0)]

    # Handle data release gaps by forward-filling macro attributes to ensure temporal consistency
    cleaned_df['Credit_Spread'] = cleaned_df['Credit_Spread'].ffill().bfill()
    cleaned_df['VIX_Close'] = cleaned_df['VIX_Close'].ffill().bfill()

    # Engineer Liquidity Proxy: Volume-to-Float Index (Standardized tracking proxy)
    cleaned_df['Liquidity_Proxy'] = cleaned_df['SP100_Volume'] / 1e7

    # Calculate Daily Garman-Klass Volatility Metrics
    # Mathematical Formula:
    # \sigma_{GK}^2 = 0.5 \cdot \left(\ln\frac{H}{L}\right)^2 - (2\ln 2 - 1) \cdot \left(\ln\frac{C}{O}\right)^2
    log_hl = np.log(cleaned_df['SP100_High'] / cleaned_df['SP100_Low'])
    log_co = np.log(cleaned_df['SP100_Close'] / cleaned_df['SP100_Open'])

    gk_variance = 0.5 * (log_hl ** 2) - (2 * np.log(2) - 1) * (log_co ** 2)

    # 🔧 FIX: on days with a small high-low range but a larger close-open gap,
    # this variance estimate can go slightly negative, which used to send NaN
    # into every downstream row via sqrt() and get silently dropped. Clip to
    # zero first so those rows survive with a (correctly) near-zero volatility.
    gk_variance = gk_variance.clip(lower=0.0)
    cleaned_df['Garman_Klass_Vol'] = np.sqrt(gk_variance)

    # Remove edge artifacts caused by mathematical operations
    cleaned_df.dropna(inplace=True)

    # Enforce strict matrix layout tracking columns
    target_columns = ['SP100_Close', 'Garman_Klass_Vol', 'VIX_Close', 'Credit_Spread', 'Liquidity_Proxy']
    return cleaned_df[target_columns]

processed_data = process_and_align_pipeline(raw_data)
print("✅ Section 1.3 Success: Final Ingestion Matrix Cleaned and Filtered.")
print(f"📋 Analytical Shape: {processed_data.shape} rows/columns.")
print("\n--- SAMPLE VIEW OF PROCESSED MATRIX HEAD ---")
print(processed_data.head())


In [ ]:
# ==============================================================================
# SECTION 2.1: STATIONARITY TRANSFORMATION ENGINE
# ==============================================================================
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

def transform_to_stationarity(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies mathematical transformations to map raw I(1) fields into
    strictly stationary I(0) continuous distribution tracking fields.
    """
    print("🔄 Initializing mathematical transformation arrays...")
    stationary_df = pd.DataFrame(index=df.index)

    # 1. Target: S&P 100 Index Close -> Daily Log Returns (Δ ln P_t)
    stationary_df['SP100_Returns'] = np.log(df['SP100_Close']).diff()

    # 2. Target Proxy: Garman-Klass Trailing Volatility -> Log First-Difference (Δ ln V_t)
    # 🔧 FIX: Garman_Klass_Vol can be exactly 0.0 on a very quiet trading day
    # (after the Section 1.3 clip), and log(0) is -inf, which propagates into
    # the diff and any model that touches this column. A tiny epsilon keeps
    # the log well-defined without materially changing real observations.
    epsilon = 1e-8
    stationary_df['GK_Vol_Diff'] = np.log(df['Garman_Klass_Vol'] + epsilon).diff()

    # 3. Covariate: VIX Index -> Log First-Difference (Δ ln VIX_t)
    stationary_df['VIX_Diff'] = np.log(df['VIX_Close']).diff()

    # 4. Covariate: Credit Spread -> Absolute First-Difference (Δ S_t)
    stationary_df['Credit_Spread_Diff'] = df['Credit_Spread'].diff()

    # 5. Covariate: Liquidity Proxy -> Absolute First-Difference (Δ L_t)
    stationary_df['Liquidity_Diff'] = df['Liquidity_Proxy'].diff()

    # Eliminate the initial boundary row containing NaN artifacts from differencing
    stationary_df.dropna(inplace=True)
    return stationary_df

# Generate the stationary matrix from your active processed data frame
stationary_matrix = transform_to_stationarity(processed_data)
print(f"📊 Section 2.1 Success: Stationary Matrix formed. Dimensions: {stationary_matrix.shape}")

# ==============================================================================
# SECTION 2.2: AUTOMATED DUAL-TEST STATIONARITY VALIDATOR
# ==============================================================================
def run_dual_stationarity_suite(df: pd.DataFrame):
    """
    Executes the ADF and KPSS tests simultaneously across all transformed variables.
    Target: Reject ADF Null (p < 0.01) AND Fail to Reject KPSS Null (p > 0.05).
    """
    print("\n🔍 EXECUTING DUAL STATIONARITY TESTING SUITE (ADF vs. KPSS)...")
    print("=" * 85)
    print(f"{'Variable Node':<22} | {'ADF p-val':<10} | {'ADF Struct':<11} | {'KPSS p-val':<10} | {'KPSS Struct'}")
    print("=" * 85)

    for col in df.columns:
        # 1. Augmented Dickey-Fuller Test (Maxlag automatically targeted via AIC optimization)
        adf_res = adfuller(df[col], autolag='AIC')
        adf_p = adf_res[1]
        adf_status = "STATIONARY" if adf_p < 0.01 else "UNIT-ROOT"

        # 2. KPSS Test (Evaluated assuming trend-stationarity constant 'c')
        kpss_res = kpss(df[col], regression='c', nlags='auto')
        kpss_p = kpss_res[1]
        kpss_status = "STATIONARY" if kpss_p > 0.05 else "UNIT-ROOT"

        print(f"{col:<22} | {adf_p:<10.4e} | {adf_status:<11} | {kpss_p:<10.4e} | {kpss_status}")
    print("=" * 85)
    print("ℹ️ Note: If both tests read 'STATIONARY', your data is ready for causal evaluation.")

# Run the validation matrix
run_dual_stationarity_suite(stationary_matrix)

# ==============================================================================
# SECTION 2.3: LINEAR DEPENDENCY & CONDITIONAL HETEROSKEDASTICITY AUDITING
# ==============================================================================
def perform_dependency_diagnostics(df: pd.DataFrame):
    """
    Runs joint portmanteau tests and Lagrange Multiplier tests to check for serial
    correlation and variance memory.
    """
    print("\n📈 MEASURING TIME-SERIES LINEAR & VOLATILITY DEPENDENCIES...")
    print("-" * 85)

    # 1. Ljung-Box Test on your Trailing Volatility Differentials (Evaluating 10 Lags)
    lb_res = acorr_ljungbox(df['GK_Vol_Diff'], lags=[10], return_df=True)
    lb_p = lb_res['lb_pvalue'].values[0]
    print(f"🔹 Ljung-Box Q-Test [GK_Vol_Diff] (Lag 10): P-Value = {lb_p:.5e}")
    if lb_p < 0.05:
        print("   👉 Result: Significant linear dependencies remain. Time-series memory is active.")

    # 2. Engle's ARCH Test on Mean-Centered S&P 100 Returns (Evaluating 10 Lags)
    mean_adjusted_returns = df['SP100_Returns'] - df['SP100_Returns'].mean()
    arch_res = het_arch(mean_adjusted_returns, maxlag=10)
    arch_p = arch_res[1]  # Extract Lagrange Multiplier p-value
    print(f"🔹 Engle ARCH LM-Test [SP100_Returns] (Lag 10): P-Value = {arch_p:.5e}")
    if arch_p < 0.01:
        print("   👉 Result: Severe ARCH effects confirmed. Volatility clustering is structural.")
    print("-" * 85)

# Execute diagnostics
perform_dependency_diagnostics(stationary_matrix)


In [ ]:
# ==============================================================================
# AUTOMATED DEPENDENCY CHECK & INSTALLATION
# ==============================================================================
import sys
import subprocess

try:
    from arch import arch_model
except ModuleNotFoundError:
    print("📦 Package 'arch' missing. Launching automated environment installation...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "arch", "--quiet"])
    from arch import arch_model
    print("✅ Package 'arch' successfully compiled and loaded.")

# ==============================================================================
# SECTION 3.1: OPTIMAL LAG SELECTION & AR-GARCH(1,1) FILTERING ENGINE
# ==============================================================================
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

def residualize_volatility_pipeline(stationary_df: pd.DataFrame) -> tuple:
    """
    Filters the stationary trailing volatility differential through an AR(p)-GARCH(1,1)
    framework to isolate structural, uncorrelated volatility innovations.
    """
    print("\n⚙️ INITIATING AR-GARCH RESIDUALIZATION FILTER ENGINE...")
    print("-" * 85)

    # Target series: Garman-Klass Volatility Differentials
    vol_series = stationary_df['GK_Vol_Diff'].values

    # 1. Determine optimal AR(p) lag using Bayesian Information Criterion (BIC)
    # 🔧 FIX: the old scan only checked lags 1-5 with AIC, which was landing on
    # lag 5 (the edge of the search window) and still leaving strong residual
    # autocorrelation (Ljung-Box p ~ 1e-7). Widening the search to two trading
    # weeks and using BIC (which penalizes extra lags more) finds a model that
    # actually captures the leftover weekly memory instead of stopping at the wall.
    best_bic = np.inf
    best_lag = 1

    for lag in range(1, 11):
        model = sm.tsa.AutoReg(vol_series, lags=lag, trend='c').fit()
        if model.bic < best_bic:
            best_bic = model.bic
            best_lag = lag

    print(f"🔹 BIC Lag Optimization Match: Optimal AR Memory set to Lag {best_lag}")

    # 2. Construct the combined AR(p)-GARCH(1,1) framework
    # We multiply by 100 to scale the values and prevent optimization convergence errors.
    scaled_series = vol_series * 100.0

    am = arch_model(
        scaled_series,
        mean='AR',
        lags=best_lag,
        vol='Garch',
        p=1,
        q=1,
        dist='normal'
    )

    res = am.fit(disp='off')
    print("\n📈 AR-GARCH ESTIMATION RESULTS SUMMARY:")
    print(res.summary())

    # 3. Extract the clean, standardized residuals (z_t = epsilon_t / sigma_t)
    standardized_residuals = res.resid / res.conditional_volatility

    # Re-align back into a clean Pandas Series matching the original time-index
    vol_innovations = pd.Series(standardized_residuals, index=stationary_df.index)

    return vol_innovations, res

# Run the filtering process
vol_shocks, garch_fit = residualize_volatility_pipeline(stationary_matrix)

# Assemble our clear, structural causal discovery dataframe
causal_df = pd.DataFrame(index=stationary_matrix.index)
causal_df['Vol_Innovations'] = vol_shocks
causal_df['VIX_Diff'] = stationary_matrix['VIX_Diff']
causal_df['Credit_Spread_Diff'] = stationary_matrix['Credit_Spread_Diff']
causal_df['Liquidity_Diff'] = stationary_matrix['Liquidity_Diff']

print(f"\n🚀 Section 3.1 Success: Causal Discovery Matrix Compiled. Dimensions: {causal_df.shape}")

# ==============================================================================
# SECTION 3.2: POST-RESIDUALIZATION DIAGNOSTIC VALIDATION (PATCHED FOR AR LAGS)
# ==============================================================================
def verify_residual_purity(df: pd.DataFrame):
    """
    Executes post-estimation tests to verify that our volatility innovations
    behave as strict mathematical white noise. Removes lag artifacts safely.
    """
    print("\n🧪 EXECUTING POST-RESIDUALIZATION DIAGNOSTIC CHECKS...")
    print("=" * 85)

    clean_series = df['Vol_Innovations'].dropna()
    clean_series = clean_series[np.isfinite(clean_series)]

    # 1. Check for remaining linear correlation in residuals via Ljung-Box (Lag=10)
    lb_res = acorr_ljungbox(clean_series, lags=10, return_df=True)
    lb_p = lb_res['lb_pvalue'].iloc[-1]
    print(f"🔹 Residual Ljung-Box Test (10 Lags Joint P-val): P-value = {lb_p:.5e}")

    # 2. Check for remaining ARCH conditional variance effects (Lag=10)
    arch_res = het_arch(clean_series, maxlag=10)
    arch_p = arch_res[1]
    print(f"🔹 Residual Engle ARCH Test (10 Lags LM P-val): P-value = {arch_p:.5e}")
    print("=" * 85)

    if lb_p > 0.05 and arch_p > 0.05:
        print("✅ SUCCESS: Residuals are strictly white noise. Internal time-series memory is fully stripped.")
        print("👉 The dataset is mathematically optimized for structural causal inference.")
    else:
        print("⚠️ WARNING: Residual dependencies remain. Consider expanding GARCH or lag specifications.")

# Run the patched diagnostic suite
verify_residual_purity(causal_df)


### Reusable causal-discovery and frozen-strategy functions

In [ ]:
# ==============================================================================
# CAUSAL DISCOVERY ENGINE (function defs only — identical methodology throughout)
# ==============================================================================
import numpy as np
import pandas as pd
import statsmodels.api as sm

def execute_structural_causal_discovery(df: pd.DataFrame, alpha_thresh: float = 0.05) -> dict:
    """Fits the causal graph on whatever slice of data is passed in (TRAIN only,
    in every use below — never sees validation or test rows)."""
    clean_causal_df = df.dropna().copy()
    var_names = list(clean_causal_df.columns)
    target_var = 'Vol_Innovations'
    target_idx = var_names.index(target_var)

    tau_max = 5
    n_nodes = len(var_names)
    p_matrix = np.ones((n_nodes, n_nodes, tau_max + 1))
    val_matrix = np.zeros((n_nodes, n_nodes, tau_max + 1))

    y_series = clean_causal_df[target_var].values

    for source_idx, source_name in enumerate(var_names):
        for tau in range(1, tau_max + 1):
            x_shifted = clean_causal_df[source_name].shift(tau).values
            X_matrix = [x_shifted]
            for lag in range(1, tau_max + 1):
                X_matrix.append(clean_causal_df[target_var].shift(lag).values)
            X_matrix = np.column_stack(X_matrix)

            valid_mask = np.all(np.isfinite(X_matrix), axis=1) & np.isfinite(y_series)
            if np.sum(valid_mask) > 50:
                y_cropped = y_series[valid_mask]
                X_cropped = X_matrix[valid_mask]
                X_cropped_with_c = sm.add_constant(X_cropped)
                model = sm.OLS(y_cropped, X_cropped_with_c).fit()
                p_matrix[source_idx, target_idx, tau] = model.pvalues[1]
                val_matrix[source_idx, target_idx, tau] = model.params[1]

    return {
        'p_matrix': p_matrix, 'val_matrix': val_matrix,
        'var_names': var_names, 'clean_df': clean_causal_df, 'tau_max': tau_max
    }

def analyze_causal_pathways(outputs: dict, label: str = ""):
    var_names = outputs['var_names']
    p_matrix = outputs['p_matrix']
    val_matrix = outputs['val_matrix']
    tau_max = outputs['tau_max']
    target_idx = var_names.index('Vol_Innovations')

    print(f"🔍 Causal parents discovered {label}:")
    found = False
    for source_idx, source_name in enumerate(var_names):
        for tau in range(1, tau_max + 1):
            p_val = p_matrix[source_idx, target_idx, tau]
            if p_val < 0.05:
                found = True
                print(f"   {source_name} (lag {tau}): beta={val_matrix[source_idx, target_idx, tau]:.4f}, p={p_val:.3e}")
    if not found:
        print("   ⚠️ No significant causal pathways found on this data slice.")


In [ ]:
# ==============================================================================
# FROZEN v4 STRATEGY ENGINE (function defs only — thresholds NOT re-tuned here)
# ==============================================================================
import numpy as np
import pandas as pd

def construct_causal_multiplier(df_causal_clean: pd.DataFrame, outputs: dict,
                                  baseline_multiplier: float = 2.0, min_multiplier: float = 1.3) -> pd.Series:
    var_names = outputs['var_names']
    p_matrix = outputs['p_matrix']
    val_matrix = outputs['val_matrix']
    tau_max = outputs['tau_max']
    target_idx = var_names.index('Vol_Innovations')
    n = len(df_causal_clean)

    composite_risk = np.zeros(n)
    active_channels = 0
    for source_idx, source_name in enumerate(var_names):
        for tau in range(1, tau_max + 1):
            p_val = p_matrix[source_idx, target_idx, tau]
            beta = val_matrix[source_idx, target_idx, tau]
            if p_val < 0.05:
                shock_series = df_causal_clean[source_name].shift(tau).fillna(0).values
                shock_std = shock_series.std()
                z_shock = shock_series / shock_std if shock_std > 1e-12 else np.zeros(n)
                composite_risk += np.abs(beta) * np.abs(z_shock)
                active_channels += 1
    if active_channels > 0:
        composite_risk /= active_channels

    risk_series = pd.Series(composite_risk, index=df_causal_clean.index)
    smoothed_risk = risk_series.ewm(span=10, min_periods=1).mean()
    danger_pct = smoothed_risk.rolling(252, min_periods=20).apply(
        lambda w: (w.iloc[-1] > w).mean(), raw=False
    ).fillna(0.0).values

    dynamic_multipliers = baseline_multiplier - (baseline_multiplier - min_multiplier) * danger_pct
    dynamic_multipliers = np.clip(dynamic_multipliers, min_multiplier, baseline_multiplier)
    return pd.Series(dynamic_multipliers, index=df_causal_clean.index)


def execute_causal_trailing_stop(df_raw: pd.DataFrame, df_processed: pd.DataFrame, lambda_t: pd.Series) -> pd.DataFrame:
    common_index = df_processed.index.intersection(lambda_t.index)
    prices = df_raw.loc[common_index]
    processed = df_processed.loc[common_index]
    multipliers = lambda_t.loc[common_index]

    backtest_df = pd.DataFrame(index=common_index)
    backtest_df['Close'] = prices['SP100_Close']
    backtest_df['Base_Vol'] = processed['Garman_Klass_Vol'].rolling(5).mean().bfill()
    backtest_df['Causal_Multiplier'] = multipliers
    backtest_df['Close_MA5'] = backtest_df['Close'].rolling(5, min_periods=1).mean()

    standard_stop_arr = np.zeros(len(backtest_df))
    causal_stop_arr = np.zeros(len(backtest_df))
    standard_state = np.ones(len(backtest_df))
    causal_state = np.ones(len(backtest_df))
    static_multiplier = 2.0

    for i in range(len(backtest_df)):
        current_close = backtest_df['Close'].iloc[i]
        current_vol = backtest_df['Base_Vol'].iloc[i]
        current_causal_mult = backtest_df['Causal_Multiplier'].iloc[i]

        raw_standard_stop = current_close - (static_multiplier * current_vol * current_close)
        raw_causal_stop = current_close - (current_causal_mult * current_vol * current_close)

        if i == 0:
            standard_stop_arr[i] = raw_standard_stop
            causal_stop_arr[i] = raw_causal_stop
        else:
            if standard_state[i-1] == 1:
                standard_stop_arr[i] = max(standard_stop_arr[i-1], raw_standard_stop)
                if current_close < standard_stop_arr[i]:
                    standard_state[i] = 0
                    standard_stop_arr[i] = raw_standard_stop
                else:
                    standard_state[i] = 1
            else:
                if current_close > backtest_df['Close'].iloc[i-1]:
                    standard_state[i] = 1
                    standard_stop_arr[i] = raw_standard_stop
                else:
                    standard_state[i] = 0
                    standard_stop_arr[i] = standard_stop_arr[i-1]

            if causal_state[i-1] == 1:
                causal_stop_arr[i] = max(causal_stop_arr[i-1], raw_causal_stop)
                if current_close < causal_stop_arr[i]:
                    causal_state[i] = 0
                    causal_stop_arr[i] = raw_causal_stop
                else:
                    causal_state[i] = 1
            else:
                calm_threshold = 1.8
                if current_causal_mult >= calm_threshold:
                    reentry_trigger = current_close > backtest_df['Close'].iloc[i-1]
                else:
                    reentry_trigger = current_close > backtest_df['Close_MA5'].iloc[i-1]
                if reentry_trigger:
                    causal_state[i] = 1
                    causal_stop_arr[i] = raw_causal_stop
                else:
                    causal_state[i] = 0
                    causal_stop_arr[i] = causal_stop_arr[i-1]

    backtest_df['Standard_Stop_Level'] = standard_stop_arr
    backtest_df['Causal_Stop_Level'] = causal_stop_arr
    backtest_df['Standard_State'] = standard_state
    backtest_df['Causal_State'] = causal_state
    return backtest_df


def compute_comprehensive_risk_metrics(backtest_df: pd.DataFrame) -> pd.DataFrame:
    metrics = {}
    close_prices = backtest_df['Close'].values
    market_returns = np.diff(np.log(close_prices))
    strategies = {'Standard_Baseline': 'Standard_State', 'Causal_Adaptive': 'Causal_State'}

    for name, state_col in strategies.items():
        state = backtest_df[state_col].values
        stop_count = np.sum((state[:-1] == 1) & (state[1:] == 0))
        strat_returns = market_returns * state[:-1]

        cum_wealth = np.exp(np.cumsum(strat_returns))
        total_ret = cum_wealth[-1] - 1 if len(cum_wealth) > 0 else 0
        years = len(strat_returns) / 252.0
        ann_return = (total_ret + 1) ** (1.0 / years) - 1 if years > 0 else 0
        ann_vol = np.std(strat_returns) * np.sqrt(252)
        sharpe = ann_return / ann_vol if ann_vol > 0 else 0

        running_max = np.maximum.accumulate(cum_wealth)
        running_max[running_max == 0] = 1.0
        drawdowns = (cum_wealth - running_max) / running_max
        max_dd = np.min(drawdowns) if len(drawdowns) > 0 else 0

        metrics[name] = {
            'Annualised Return': ann_return, 'Annualised Volatility': ann_vol,
            'Sharpe Ratio': sharpe, 'Maximum Drawdown': max_dd, 'Total Stop-Out Events': stop_count
        }
    return pd.DataFrame(metrics).T

def fmt(df):
    return df.to_string(formatters={
        'Annualised Return': '{:,.2%}'.format, 'Annualised Volatility': '{:,.2%}'.format,
        'Sharpe Ratio': '{:,.3f}'.format, 'Maximum Drawdown': '{:,.2%}'.format,
        'Total Stop-Out Events': '{:,.0f}'.format
    })


### Walk-forward folds

In [ ]:
# ==============================================================================
# WALK-FORWARD: expanding-window TRAIN -> TEST folds
# Each fold: causal discovery fit on TRAIN (0 : train_end) only, applied
# with FROZEN v4 thresholds, tested purely on the next untouched segment.
# ==============================================================================
full_clean_causal_df = causal_df.dropna().copy()
n = len(full_clean_causal_df)

min_train_size = 750   # ~3 years minimum history before the first fold fits
n_folds = 5
usable = n - min_train_size
fold_size = usable // n_folds

fold_results = []

for k in range(n_folds):
    train_end = min_train_size + k * fold_size
    test_start = train_end
    test_end = n if k == n_folds - 1 else train_end + fold_size

    train_slice = full_clean_causal_df.iloc[0:train_end]
    test_index = full_clean_causal_df.iloc[test_start:test_end].index

    print(f"\n{'='*75}\nFOLD {k+1}/{n_folds}  |  TRAIN: {train_slice.index[0].date()} to {train_slice.index[-1].date()} "
          f"({len(train_slice)} rows)  |  TEST: {test_index[0].date()} to {test_index[-1].date()} ({len(test_index)} rows)")
    print("=" * 75)

    causal_outputs_fold = execute_structural_causal_discovery(train_slice)
    analyze_causal_pathways(causal_outputs_fold, label=f"(Fold {k+1} TRAIN only)")

    multiplier_fold = construct_causal_multiplier(
        full_clean_causal_df, causal_outputs_fold, baseline_multiplier=2.0, min_multiplier=1.3
    )
    test_multiplier = multiplier_fold.loc[test_index]

    backtest_fold = execute_causal_trailing_stop(raw_data, processed_data, test_multiplier)
    metrics_fold = compute_comprehensive_risk_metrics(backtest_fold)

    print(fmt(metrics_fold))

    fold_results.append({
        'fold': k + 1,
        'test_start': test_index[0].date(), 'test_end': test_index[-1].date(),
        'baseline_sharpe': metrics_fold.loc['Standard_Baseline', 'Sharpe Ratio'],
        'causal_sharpe': metrics_fold.loc['Causal_Adaptive', 'Sharpe Ratio'],
        'baseline_dd': metrics_fold.loc['Standard_Baseline', 'Maximum Drawdown'],
        'causal_dd': metrics_fold.loc['Causal_Adaptive', 'Maximum Drawdown'],
    })

# ==============================================================================
# AGGREGATE SUMMARY ACROSS ALL FOLDS
# ==============================================================================
summary_df = pd.DataFrame(fold_results).set_index('fold')
print("\n" + "=" * 75)
print("📊 PER-FOLD SUMMARY (out-of-sample results only)")
print("=" * 75)
print(summary_df.to_string(formatters={
    'baseline_sharpe': '{:.3f}'.format, 'causal_sharpe': '{:.3f}'.format,
    'baseline_dd': '{:.2%}'.format, 'causal_dd': '{:.2%}'.format
}))

sharpe_win_rate = (summary_df['causal_sharpe'] > summary_df['baseline_sharpe']).mean()
dd_win_rate = (summary_df['causal_dd'] > summary_df['baseline_dd']).mean()

print("\n" + "-" * 75)
print(f"Causal_Adaptive beat baseline on Sharpe in {sharpe_win_rate:.0%} of folds")
print(f"Causal_Adaptive beat baseline on Drawdown in {dd_win_rate:.0%} of folds")
print(f"Mean Sharpe — Causal_Adaptive: {summary_df['causal_sharpe'].mean():.3f} | Standard_Baseline: {summary_df['baseline_sharpe'].mean():.3f}")
print(f"Mean Max DD — Causal_Adaptive: {summary_df['causal_dd'].mean():.2%} | Standard_Baseline: {summary_df['baseline_dd'].mean():.2%}")
print("-" * 75)
if sharpe_win_rate >= 0.6 and dd_win_rate >= 0.6:
    print("🌟 Consistent edge across most folds — this is meaningfully stronger evidence")
    print("   than a single train/test split.")
else:
    print("⚠️ Wins are inconsistent across folds — treat any single-split result")
    print("   (in-sample or otherwise) with caution; report the fold spread, not just the average.")
